# 字节对编码标记化

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/


In [2]:
corpus = [
    "This is the Hugging Face course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [4]:
from collections import defaultdict

word_freqs = defaultdict(int)

for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print(word_freqs)

defaultdict(<class 'int'>, {'This': 3, 'Ġis': 2, 'Ġthe': 1, 'ĠHugging': 1, 'ĠFace': 1, 'Ġcourse': 1, '.': 4, 'Ġchapter': 1, 'Ġabout': 1, 'Ġtokenization': 1, 'Ġsection': 1, 'Ġshows': 1, 'Ġseveral': 1, 'Ġtokenizer': 1, 'Ġalgorithms': 1, 'Hopefully': 1, ',': 1, 'Ġyou': 1, 'Ġwill': 1, 'Ġbe': 1, 'Ġable': 1, 'Ġto': 1, 'Ġunderstand': 1, 'Ġhow': 1, 'Ġthey': 1, 'Ġare': 1, 'Ġtrained': 1, 'Ġand': 1, 'Ġgenerate': 1, 'Ġtokens': 1})


In [5]:
alphabet = []

for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)
alphabet.sort()

print(alphabet)

[',', '.', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ']


In [6]:
vocab = ["<|endoftext|>"] + alphabet.copy()
vocab

['<|endoftext|>',
 ',',
 '.',
 'F',
 'H',
 'T',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'y',
 'z',
 'Ġ']

In [7]:
splits = {word: [c for c in word] for word in word_freqs.keys()}
splits

{'This': ['T', 'h', 'i', 's'],
 'Ġis': ['Ġ', 'i', 's'],
 'Ġthe': ['Ġ', 't', 'h', 'e'],
 'ĠHugging': ['Ġ', 'H', 'u', 'g', 'g', 'i', 'n', 'g'],
 'ĠFace': ['Ġ', 'F', 'a', 'c', 'e'],
 'Ġcourse': ['Ġ', 'c', 'o', 'u', 'r', 's', 'e'],
 '.': ['.'],
 'Ġchapter': ['Ġ', 'c', 'h', 'a', 'p', 't', 'e', 'r'],
 'Ġabout': ['Ġ', 'a', 'b', 'o', 'u', 't'],
 'Ġtokenization': ['Ġ',
  't',
  'o',
  'k',
  'e',
  'n',
  'i',
  'z',
  'a',
  't',
  'i',
  'o',
  'n'],
 'Ġsection': ['Ġ', 's', 'e', 'c', 't', 'i', 'o', 'n'],
 'Ġshows': ['Ġ', 's', 'h', 'o', 'w', 's'],
 'Ġseveral': ['Ġ', 's', 'e', 'v', 'e', 'r', 'a', 'l'],
 'Ġtokenizer': ['Ġ', 't', 'o', 'k', 'e', 'n', 'i', 'z', 'e', 'r'],
 'Ġalgorithms': ['Ġ', 'a', 'l', 'g', 'o', 'r', 'i', 't', 'h', 'm', 's'],
 'Hopefully': ['H', 'o', 'p', 'e', 'f', 'u', 'l', 'l', 'y'],
 ',': [','],
 'Ġyou': ['Ġ', 'y', 'o', 'u'],
 'Ġwill': ['Ġ', 'w', 'i', 'l', 'l'],
 'Ġbe': ['Ġ', 'b', 'e'],
 'Ġable': ['Ġ', 'a', 'b', 'l', 'e'],
 'Ġto': ['Ġ', 't', 'o'],
 'Ġunderstand': ['Ġ', 'u', 'n'

In [8]:
def compute_pair_freqs(splits):
    """
    统计所有相邻字符对（bigram）在语料库中出现的频次。

    BPE 的核心思想：每轮找出频次最高的相邻 token 对，将其合并为新 token。
    本函数负责第一步：扫描当前所有单词的切分状态，累计每个相邻对的频次。

    参数：
        splits: dict，{word: [token, token, ...]}
                当前每个单词被切分成的 token 列表（初始为单字符）

    返回：
        pair_freqs: defaultdict，{(token_a, token_b): 频次}
                    所有相邻 token 对 → 该对在语料库中出现的总次数

    工作流程（以单词 "This" 频次=3 为例）：
        split = ['T', 'h', 'i', 's']
        相邻对：('T','h'), ('h','i'), ('i','s')
        每对各 += 3（因为 "This" 出现了 3 次）
    """
    pair_freqs = defaultdict(int)

    for word, freq in word_freqs.items():       # 遍历每个单词及其在语料库中的频次
        split = splits[word]                    # 取该单词当前的 token 切分列表
        if len(split) == 1:
            continue                            # 单字符单词没有相邻对，跳过

        for i in range(len(split) - 1):         # 滑动窗口，取每对相邻 token
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq            # 累加：该对出现次数 = 单词频次

    return pair_freqs

In [9]:
pair_freqs = compute_pair_freqs(splits)

# 统计总共有多少种相邻 token 对
print(f"共发现 {len(pair_freqs)} 种相邻 token 对\n")

# 打印频次最高的 10 对（排序后），便于直观看出哪对最值得合并
print("=== 频次最高的 10 个 token 对 ===")
sorted_pairs = sorted(pair_freqs.items(), key=lambda x: x[1], reverse=True)
for pair, freq in sorted_pairs[:10]:
    print(f"  {pair[0]!r:8} + {pair[1]!r:12} -> 频次: {freq}")

# 找出当前最佳合并候选
best_pair = sorted_pairs[0][0]
print(f"\n>>> 本轮最佳合并对: {best_pair[0]!r} + {best_pair[1]!r}，频次 = {sorted_pairs[0][1]}")

共发现 99 种相邻 token 对

=== 频次最高的 10 个 token 对 ===
  'Ġ'      + 't'          -> 频次: 7
  'i'      + 's'          -> 频次: 5
  'e'      + 'r'          -> 频次: 5
  'Ġ'      + 'a'          -> 频次: 5
  't'      + 'o'          -> 频次: 4
  'e'      + 'n'          -> 频次: 4
  'T'      + 'h'          -> 频次: 3
  'h'      + 'i'          -> 频次: 3
  't'      + 'h'          -> 频次: 3
  'o'      + 'u'          -> 频次: 3

>>> 本轮最佳合并对: 'Ġ' + 't'，频次 = 7


In [10]:
best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
        best_pair = pair
        max_freq = freq

print(best_pair, max_freq)

('Ġ', 't') 7


In [11]:
merges = {("Ġ", "t"): "Ġt"}
vocab.append("Ġt")

In [12]:
def merge_pair(a, b, splits):
    """
    将所有单词切分中出现的相邻 token 对 (a, b) 合并为单个新 token (a+b)。

    这是 BPE 训练的第二步：确定最佳合并对后，对整个 splits 执行原地替换。
    合并后 splits 中所有 [... a, b ...] 都变成 [... ab ...]，为下一轮统计做准备。

    参数：
        a:      str，待合并对的左侧 token（如 'Ġ'）
        b:      str，待合并对的右侧 token（如 't'）
        splits: dict，{word: [token, ...]}，会被原地修改

    返回：
        splits: 修改后的同一个 dict

    示例（合并 'Ġ' + 't'）：
        before: 'Ġtrained' -> ['Ġ', 't', 'r', 'a', 'i', 'n', 'e', 'd']
        after:  'Ġtrained' -> ['Ġt', 'r', 'a', 'i', 'n', 'e', 'd']

    注意：使用 while 而非 for，是因为合并后列表长度缩短，
         索引需要原地更新，且需要支持连续合并（如 aab -> 不会将 aa 合并成 aa+b）。
    """
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue                            # 单 token 单词，无需处理

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                # 找到匹配的相邻对，合并为 a+b，列表长度 -1
                split = split[:i] + [a + b] + split[i + 2:]
                # 合并后 i 不变，继续检查当前位置（避免跳过连续相同对）
            else:
                i += 1                          # 不匹配则向右移动
        splits[word] = split                    # 写回修改后的切分

    return splits

In [13]:
# 执行本轮最佳合并：将 'Ġ' + 't' 合并为 'Ġt'
# 选取几个包含 't' 的单词，对比合并前后的变化
demo_words = ['Ġtrained', 'Ġthe', 'Ġto', 'Ġtokenization', 'This']

print("=== 合并 'Ġ' + 't'  ->  'Ġt' ===\n")
print("合并前：")
for w in demo_words:
    print(f"  {w!r:20} -> {splits[w]}")

splits = merge_pair("Ġ", "t", splits)

print("\n合并后：")
for w in demo_words:
    print(f"  {w!r:20} -> {splits[w]}")

print(f"\n注意：'This' 中的 't' 不以 'Ġ' 开头，所以不受影响。")

=== 合并 'Ġ' + 't'  ->  'Ġt' ===

合并前：
  'Ġtrained'           -> ['Ġ', 't', 'r', 'a', 'i', 'n', 'e', 'd']
  'Ġthe'               -> ['Ġ', 't', 'h', 'e']
  'Ġto'                -> ['Ġ', 't', 'o']
  'Ġtokenization'      -> ['Ġ', 't', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 't', 'i', 'o', 'n']
  'This'               -> ['T', 'h', 'i', 's']

合并后：
  'Ġtrained'           -> ['Ġt', 'r', 'a', 'i', 'n', 'e', 'd']
  'Ġthe'               -> ['Ġt', 'h', 'e']
  'Ġto'                -> ['Ġt', 'o']
  'Ġtokenization'      -> ['Ġt', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 't', 'i', 'o', 'n']
  'This'               -> ['T', 'h', 'i', 's']

注意：'This' 中的 't' 不以 'Ġ' 开头，所以不受影响。


In [14]:
vocab_size = 50

while len(vocab) < vocab_size:
    pair_freqs = compute_pair_freqs(splits)
    best_pair = ""
    max_freq = None
    for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    splits = merge_pair(*best_pair, splits)
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])

In [15]:
print(merges)

{('Ġ', 't'): 'Ġt', ('i', 's'): 'is', ('e', 'r'): 'er', ('Ġ', 'a'): 'Ġa', ('Ġt', 'o'): 'Ġto', ('e', 'n'): 'en', ('T', 'h'): 'Th', ('Th', 'is'): 'This', ('o', 'u'): 'ou', ('s', 'e'): 'se', ('Ġto', 'k'): 'Ġtok', ('Ġtok', 'en'): 'Ġtoken', ('n', 'd'): 'nd', ('Ġ', 'is'): 'Ġis', ('Ġt', 'h'): 'Ġth', ('Ġth', 'e'): 'Ġthe', ('i', 'n'): 'in', ('Ġ', 'c'): 'Ġc', ('Ġa', 'b'): 'Ġab', ('Ġtoken', 'i'): 'Ġtokeni'}


In [16]:
print(vocab)

['<|endoftext|>', ',', '.', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ', 'Ġt', 'is', 'er', 'Ġa', 'Ġto', 'en', 'Th', 'This', 'ou', 'se', 'Ġtok', 'Ġtoken', 'nd', 'Ġis', 'Ġth', 'Ġthe', 'in', 'Ġc', 'Ġab', 'Ġtokeni']


In [17]:
def tokenize(text):
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]
    splits = [[l for l in word] for word in pre_tokenized_text]
    for pair, merge in merges.items():
        for idx, split in enumerate(splits):
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                else:
                    i += 1
            splits[idx] = split

    return sum(splits, [])

In [18]:
tokenize("This is not a token.")

['This', 'Ġis', 'Ġ', 'n', 'o', 't', 'Ġa', 'Ġtoken', '.']